# Machine Learning vs. Deep Learning: Foundations & Architecture

##  Difference Between Machine Learning and Deep Learning

While Deep Learning is technically a subset of Machine Learning, they differ fundamentally in how they process data and learn representations.

| Feature | Machine Learning (ML) | Deep Learning (DL) |
| :--- | :--- | :--- |
| **Approach** | Uses algorithms to parse data, learn from it, and make informed decisions. | Uses neural networks with multiple layers to model complex patterns. |
| **Feature Engineering** | **Manual:** Humans must identify and extract features (e.g., "edges" in an image). | **Automatic:** The network learns features directly from raw data. |
| **Data Requirements** | Can perform well with smaller datasets. | Requires large amounts of data to generalize effectively. |
| **Interpretability** | **High:** Models like Decision Trees are transparent. | **Low:** Often acts as a "Black Box" due to millions of parameters. |
| **Examples** | Linear Regression, SVM, Random Forests. | CNNs, RNNs, Transformers. |






**Key Difference:** DL automatically discovers the representations needed for detection or classification, whereas ML requires these features to be provided manually.

---

##  How Machine Learning Fails

Understanding failure modes is critical for building robust systems.

### A. Insufficient or Imbalanced Data
* **Lack of Data:** Models cannot learn patterns without enough representative examples.
* **Imbalance:** If 99% of data is Class A, the model will bias towards Class A, ignoring Class B.

### B. Overfitting
* **The Problem:** The model learns the *noise* in the training data rather than the underlying signal.
* **Symptom:** Performs perfectly on training data but fails miserably on new, unseen data.

### C. Underfitting
* **The Problem:** The model is too simple (e.g., trying to fit a straight line to a curved dataset).
* **Symptom:** Poor performance on both training and test data.

### D. Poor Feature Selection
* **"Garbage In, Garbage Out":** If input features are irrelevant, the model becomes confused and inaccurate.

### E. Concept Drift
* **The Problem:** Real-world patterns change over time (e.g., spam email tactics evolve).
* **Result:** A static model becomes outdated and accuracy degrades.

---

##   How a Biological Neuron Works

Deep learning is loosely inspired by the biological brain.

* **Structure:**
    `Dendrites → Cell Body (Soma) → Axon → Synapses`
* **The Process:**
    1.  **Input Reception:** Dendrites receive chemical signals from other neurons.
    2.  **Integration:** The Soma sums all incoming signals.
    3.  **Threshold Check:** If the sum exceeds a specific voltage threshold, an **Action Potential** is fired.
    4.  **Transmission:** The electrical impulse travels down the Axon to the Synapses to trigger the next neuron.



---

##   Introduction to Artificial Neurons

We mathematically model the biological process using the **Perceptron**.

### Basic Artificial Neuron Model
`Inputs (x) → Weights (w) → Summation (z) → Activation (f) → Output (y)`

### Mathematical Representation
1.  **Linear Combination:**
    $$z = \sum (w_i \cdot x_i) + b$$
2.  **Activation:**
    $$y = f(z)$$

**Where:**
* $x_i$: Input features.
* $w_i$: Weights (importance of each feature).
* $b$: Bias (allows shifting the activation threshold).
* $f()$: Activation function (introduces non-linearity).

---

##  Perceptron Architecture

The Perceptron is the simplest form of a neural network (a single-layer network).

### Key Components
1.  **Input Layer:** Receives raw feature vectors.
2.  **Weights:** Adjustable parameters learned during training.
3.  **Bias:** Offset value.
4.  **Activation Function:** Typically a **Step Function** (0 or 1) for binary classification.

### The Learning Rule (Perceptron Algorithm)
For each training example $(x, target)$:
1.  **Predict:** `prediction = activation(dot(w, x) + b)`
2.  **Calculate Error:** `error = target - prediction`
3.  **Update Weights:** `w = w + (learning_rate * error * x)`
4.  **Update Bias:** `b = b + (learning_rate * error)`

 **Limitation:** Perceptrons can only solve **Linearly Separable** problems. They cannot solve the XOR problem (Minsky & Papert, 1969).







In [ ]:


import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class Perceptron:
    def __init__(self, learning_rate=0.01, n_iters=1000):
        self.lr = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.errors_history = []
    
    def activation(self, x):
        # Step function: Returns 1 if x >= 0, else 0
        return np.where(x >= 0, 1, 0)
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Ensure y is binary (0 or 1)
        y_ = np.where(y == 0, 0, 1)
        
        # Training loop
        for _ in range(self.n_iters):
            total_error = 0
            for idx, x_i in enumerate(X):
                # Forward pass
                linear_output = np.dot(x_i, self.weights) + self.bias
                y_predicted = self.activation(linear_output)
                
                # Calculate error
                error = y_[idx] - y_predicted
                total_error += abs(error)
                
                # Update weights and bias
                self.weights += self.lr * error * x_i
                self.bias += self.lr * error
            
            self.errors_history.append(total_error)
            
            # Early stopping if converged (0 errors)
            if total_error == 0:
                break
    
    def predict(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return self.activation(linear_output)
    
    def accuracy(self, X, y):
        y_ = np.where(y == 0, 0, 1)
        predictions = self.predict(X)
        return np.mean(predictions == y_)

# --- Execution ---

# 1. Load and prepare data
iris = load_iris()
# Use only the first 100 samples (2 classes) and first 2 features for visualization
X = iris.data[:100, :2]  
y = iris.target[:100]    

# 2. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Standardize features (Critical for convergence)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 4. Create and train perceptron
perceptron = Perceptron(learning_rate=0.1, n_iters=100)
perceptron.fit(X_train, y_train)

# 5. Evaluate
train_acc = perceptron.accuracy(X_train, y_train)
test_acc = perceptron.accuracy(X_test, y_test)

print(f"Training Accuracy: {train_acc:.2f}")
print(f"Test Accuracy: {test_acc:.2f}")

# 6. Visualization Functions
def plot_decision_boundary(X, y, model, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='coolwarm')
    plt.xlabel('Sepal Length (standardized)')
    plt.ylabel('Sepal Width (standardized)')
    plt.title(title)
    plt.show()

# Plot Decision Boundary
plot_decision_boundary(X_train, y_train, perceptron, 
                       "Perceptron Decision Boundary (Training Set)")

# Plot Error Convergence
plt.figure(figsize=(10, 4))
plt.plot(range(len(perceptron.errors_history)), perceptron.errors_history)
plt.xlabel('Iterations')
plt.ylabel('Total Error')
plt.title('Error Convergence During Training')
plt.grid(True)
plt.show()